<a href="https://colab.research.google.com/github/rockers2004/LP-V/blob/main/HPC_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving input4.jpg to input4.jpg


In [ ]:
%%writefile pixel_sort.cu

#include <stdio.h>
#include <stdlib.h>
#include <vector>
#include <algorithm>
#include <numeric> // For std::iota
#include <opencv2/opencv.hpp>
#include <cuda_runtime.h>

using namespace cv;
using namespace std;

// CUDA Kernel (Odd-Even Sort)
// Sorts integers. In this case, encoded brightness and original index.
__global__ void oddEvenSort(int *data, int n, int phase)
{
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    int i = idx * 2 + phase;

    if (i + 1 < n)
    {
        if (data[i] > data[i + 1])
        {
            int temp = data[i];
            data[i] = data[i + 1];
            data[i + 1] = temp;
        }
    }
}

int main()
{
    // Load COLOR image
    Mat img = imread("images.jpg", IMREAD_COLOR);

    if (img.empty())
    {
        printf("Image not found!\n");
        return -1;
    }

    int rows = img.rows;
    int cols = img.cols;

    // Define block size for CUDA kernel
    int blockSize = 256;

    // Process each COLUMN independently (VERTICAL EFFECT)
    for (int c = 0; c < cols; c++)
    {
        vector<int> brightness_values;       // Stores brightness for pixels meeting threshold
        vector<Vec3b> actual_pixels;         // Stores the actual Vec3b values for pixels meeting threshold
        vector<int> target_image_rows;       // Stores original row indices in the image for these pixels

        // Collect pixels in column that meet the brightness threshold
        for (int r = 0; r < rows; r++)
        {
            Vec3b pixel = img.at<Vec3b>(r, c);

            int b = pixel[0];
            int g = pixel[1];
            int red = pixel[2];

            int bright = b + g + red;

            if (bright > 300)   // 🔥 tweak threshold
            {
                brightness_values.push_back(bright);
                actual_pixels.push_back(pixel);
                target_image_rows.push_back(r);
            }
        }

        int n = brightness_values.size();
        if (n == 0) continue; // Skip if no pixels meet the criteria in this column

        // Encode brightness and original_sub_index into a single int for GPU sorting.
        // Brightness (max 255*3 = 765) in upper bits, original_sub_index (up to n-1) in lower bits.
        // Assuming n < 65536 for original_sub_index to fit in 16 bits.
        vector<int> encoded_data(n);
        for (int i = 0; i < n; ++i)
        {
            // Shift brightness left by 16 bits, then OR with the original_sub_index (i).
            // This makes brightness the primary sort key, with original_sub_index as tie-breaker.
            encoded_data[i] = (brightness_values[i] << 16) | (i & 0xFFFF);
        }

        // Allocate GPU memory for encoded data
        int *d_data;
        cudaMalloc((void**)&d_data, n * sizeof(int));

        // Copy encoded data from host to device
        cudaMemcpy(d_data, encoded_data.data(), n * sizeof(int), cudaMemcpyHostToDevice);

        // Calculate grid size for GPU kernel
        int gridSize = (n + (2 * blockSize) - 1) / (2 * blockSize); // Corrected grid size calculation
        if (gridSize == 0) gridSize = 1; // Ensure at least one block for small N

        // GPU sorting on the encoded data
        for (int phase = 0; phase < n; phase++)
        {
            oddEvenSort<<<gridSize, blockSize>>>(d_data, n, phase % 2);
            cudaDeviceSynchronize();
        }

        // Copy sorted encoded data back to host
        cudaMemcpy(encoded_data.data(), d_data, n * sizeof(int), cudaMemcpyDeviceToHost);

        // Free GPU memory
        cudaFree(d_data);

        // Reconstruct sorted pixels on CPU based on the sorted encoded_data
        vector<Vec3b> sorted_pixels_for_column(n);
        for (int i = 0; i < n; ++i)
        {
            // Extract the original sub-index from the sorted encoded value
            int original_sub_index = encoded_data[i] & 0xFFFF;
            sorted_pixels_for_column[i] = actual_pixels[original_sub_index];
        }

        // Put the sorted pixels back into the image column at their original filtered positions
        for (int i = 0; i < n; i++)
        {
            img.at<Vec3b>(target_image_rows[i], c) = sorted_pixels_for_column[i];
        }
    }

    // Save output
    imwrite("sorted.png", img);
    printf("Artistic sorted image saved as sorted.png\n");

    return 0;
}

Overwriting pixel_sort.cu


In [ ]:

!apt-get update

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy

In [ ]:
!apt-get install -y libopencv-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas gstreamer1.0-plugins-base
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libavcodec-dev
  libavformat-dev libavutil-dev libcdparanoia0 libcharls2 libdc1394-dev
  libdouble-conversion3 libexif-dev libexif-doc libexif12 libgdcm-dev
  libgdcm3.0 libgl2ps1.4 libglew2.2 libgphoto2-6 libgphoto2-dev
  libgphoto2-l10n libgphoto2-port12 libgstreamer-plugins-base1.0-0 libgtk-3-0
  libgtk-3-bin libgtk-3-common libilmbase-dev libilmbase25
  libopencv-calib3d-dev libopencv-calib3d4.5d libopencv-contrib-dev
  libopencv-contrib4.5d libopencv-core-dev libopencv-core4.5d
  libopencv-dnn-dev libopencv-dnn4.5d libopencv-features2d-dev
  libopencv-features2d4.5d libopencv-flann-dev libopencv-flann4.5d
  libopencv-highgui-dev libopencv-highgui4.5d libopencv-imgcodecs-dev
  libopencv-imgcodecs4.5d libopen

In [ ]:
!nvcc pixel_sort.cu -o pixel_sort `pkg-config --cflags --libs opencv4`

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::buildMaps" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::warp" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(100): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is only partially overridden in class "cv::detail::FeatherBlender"
  clas

In [ ]:
!./pixel_sort

Artistic sorted image saved as sorted.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')